In [ ]:
import os
import sys
import geopandas as gpd
from pathlib import Path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(project_root, "src"))
from patches import build_macro_index,generate_macro_patch_for_site,generate_masks_from_macro_catalog,sample_jitter_window,read_crop_image_and_mask,ConstructionJitterDataset, make_site_splits,filter_macro_catalog
from argparse import Namespace
from config import DATA_RAW, DATA_PROCESSED, DATA_QUADS

In [ ]:
quad_catalog_path = f"{DATA_PROCESSED}/gpkg/quad_catalog.gpkg"
point_quad_path=f"{DATA_PROCESSED}/gpkg/point_quad.gpkg"

quad_catalog=gpd.read_file(quad_catalog_path)
point_quad=gpd.read_file(point_quad_path)

macro_index = build_macro_index(point_quad, quad_catalog, id_col="order")

In [ ]:
macro_index.head()
len(macro_index)
macro_index.columns

In [ ]:
MACRO_IMG_DIR = Path(DATA_PROCESSED) / "cropped_images/macro/images"
MACRO_SIZE_M = 1500  # 1 km

row = macro_index.iloc[0]

res = generate_macro_patch_for_site(
    row=row,
    macro_size_m=MACRO_SIZE_M,
    images_dir=MACRO_IMG_DIR,
    overwrite=False,
)

res


In [ ]:
from patches import generate_all_macro_patches

MACRO_IMG_DIR = Path(DATA_PROCESSED) / "cropped_images/macro/images"
MACRO_CSV = Path(DATA_PROCESSED) / "cropped_images/macro_catalog.csv"
MACRO_GPKG = Path(DATA_PROCESSED) / "gpkg/macro_catalog.gpkg"

macro_catalog = generate_all_macro_patches(
    macro_index=macro_index,
    macro_size_m=MACRO_SIZE_M,
    images_dir=MACRO_IMG_DIR,
    catalog_csv_path=MACRO_CSV,
    catalog_gpkg_path=MACRO_GPKG,
    overwrite=False,
)

In [ ]:
from pathlib import Path
import fiona
from config import DATA_PROCESSED

MACRO_IMG_DIR = Path(DATA_PROCESSED) / "cropped_images/macro/images"

for site_dir in MACRO_IMG_DIR.iterdir():
    if not site_dir.is_dir():
        continue

    order = site_dir.name
    mask_dir = site_dir / "mask"
    mask_dir.mkdir(exist_ok=True)

    gpkg_name = f"mask_{order}.gpkg"
    gpkg_path = mask_dir / gpkg_name

    if gpkg_path.exists():
        print(f"[EXISTS] {gpkg_path}, skipping")
        continue

    schema = {
        "geometry": "Polygon",
        "properties": {
            "label_date": "str:7",  # YYYY_MM
        },
    }

    with fiona.open(
        gpkg_path,
        mode="w",
        driver="GPKG",
        layer="mask",
        schema=schema,
        crs="EPSG:3857",
    ):
        pass

    print(f"[CREATED] {gpkg_path}")

### After the labeling process was completed, now those polygons were rasterized

In [ ]:
import pandas as pd
from pathlib import Path
from config import DATA_PROCESSED

macro_csv = Path(DATA_PROCESSED) / "cropped_images/macro_catalog.csv"

log_all = generate_masks_from_macro_catalog(
    macro_catalog_csv=macro_csv,
    processed_root=Path(DATA_PROCESSED),
    overwrite=True,
    save_log_path=Path(DATA_PROCESSED) / "logs/mask_rasterization_full.csv",
)

log_all["status"].value_counts()

#### Función 1: elegir la ventana 256×256 (pos/neg con jitter) y una celda de prueba. Sin Dataset todavía.

Lo que esta función hace

Input: mask (numpy 2D), crop_size=256, jitter_radius=20, pos_fraction=0.7

Output: Window (x0, y0, w, h) + metadata (pos/neg, intentos, pixeles positivos dentro del crop)

Comentarios en inglés, pocos.

In [ ]:
from pathlib import Path
import rasterio
import numpy as np

# Pick one mask tif you already generated (314x314)
mask_path = Path("/home/student/Documents/jaar/GIT/data/processed/cropped_images/macro/labels/145/mask_145_2021_03.tif")

with rasterio.open(mask_path) as src:
    mask = src.read(1)

print("Mask shape:", mask.shape, "unique:", np.unique(mask)[:10])

win = sample_jitter_window(mask, crop_size=256, jitter_radius=20, pos_fraction=0.7)
print(win)

crop_mask = mask[win.y0:win.y0+win.h, win.x0:win.x0+win.w]
print("Crop shape:", crop_mask.shape, "pos pixels:", int((crop_mask > 0).sum()))


#### Función 2: leer y recortar imagen + máscara (256×256)

Lee el patch multibanda (PlanetScope)

Lee la máscara (1 banda)

Aplica la misma CropWindow

Devuelve:

img_crop en formato (C, 256, 256) float32

mask_crop en (256, 256) int64 (ideal para CrossEntropyLoss / BCE según tu setup)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import rasterio

image_path = Path("/home/student/Documents/jaar/GIT/data/processed/cropped_images/macro/images/145/cs_145_2024_08.tif")
mask_path  = Path("/home/student/Documents/jaar/GIT/data/processed/cropped_images/macro/labels/145/mask_145_2024_08.tif")

with rasterio.open(mask_path) as src:
    mask = src.read(1)

win = sample_jitter_window(mask, crop_size=256, jitter_radius=20, pos_fraction=0.7)

img_crop, mask_crop = read_crop_image_and_mask(image_path, mask_path, win)

print("Image crop shape:", img_crop.shape, img_crop.dtype)  # (C, 256, 256)
print("Mask crop shape:", mask_crop.shape, np.unique(mask_crop), mask_crop.dtype)  # (256, 256) [0 1]
print("Window:", win)

rgb_642 = np.stack(
    [
        img_crop[5],  # Band 6 → R
        img_crop[3],  # Band 4 → G
        img_crop[1],  # Band 2 → B
    ],
    axis=-1
)

# Optional: simple normalization for visualization
rgb_642 = np.clip(rgb_642, 0, np.percentile(rgb_642, 99))
rgb_642 = rgb_642 / rgb_642.max()

i = 0
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(rgb_642)
plt.title("Image RGB")

plt.subplot(1,2,2)
plt.imshow(mask_crop, cmap="gray")
plt.title("Mask")
plt.show()



Qué va a hacer el Dataset (exactamente)

Para cada __getitem__:

Leer una fila del macro_catalog.csv

Obtener:

image_path

order

year_month

Construir mask_path correspondiente

Leer solo la máscara → decidir crop con:

70% positivo (jitter)

30% negativo

Leer imagen + máscara

Retornar:

image: (8, 256, 256) → float32

mask: (256, 256) → long (0/1)

Nada se guarda a disco.

In [ ]:
from torch.utils.data import DataLoader
from config import DATA_PROCESSED

ds = ConstructionJitterDataset(
    macro_catalog_csv=Path(DATA_PROCESSED) / "cropped_images/macro_catalog.csv",
    processed_root=Path(DATA_PROCESSED),
    crop_size=256,
    jitter_radius=20,
    pos_fraction=0.7,
    return_meta=False,
)

img, mask = ds[0]
print(img.shape, img.dtype)   # (8, 256, 256)
print(mask.shape, mask.dtype, mask.unique())

In [ ]:
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=4)

imgs, masks = next(iter(loader))
print(imgs.shape, masks.shape)

In [ ]:
from pathlib import Path
from config import DATA_PROCESSED
import pandas as pd

macro_csv = Path(DATA_PROCESSED) / "cropped_images/macro_catalog.csv"
out_csv = Path(DATA_PROCESSED) / "site_splits.csv"

splits_df = make_site_splits(
    macro_catalog_csv=macro_csv,
    out_splits_csv=out_csv,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42
)

print(splits_df["split"].value_counts())

# Verify how many rows per split (months)
df = pd.read_csv(macro_csv)
df2 = df.merge(splits_df, on="order", how="left")
print(df2["split"].value_counts())


In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from config import DATA_PROCESSED

macro_csv  = Path(DATA_PROCESSED) / "cropped_images/macro_catalog.csv"
splits_csv = Path(DATA_PROCESSED) / "site_splits.csv"

train_ds = ConstructionJitterDataset(
    macro_catalog_csv=macro_csv,
    processed_root=Path(DATA_PROCESSED),
    splits_csv=splits_csv,
    split="train",
    crop_size=256,
    jitter_radius=20,
    pos_fraction=0.7,
    return_meta=False,
)

val_ds = ConstructionJitterDataset(
    macro_catalog_csv=macro_csv,
    processed_root=Path(DATA_PROCESSED),
    splits_csv=splits_csv,
    split="val",
    crop_size=256,
    jitter_radius=20,
    pos_fraction=0.7,
    return_meta=False,
)

test_ds = ConstructionJitterDataset(
    macro_catalog_csv=macro_csv,
    processed_root=Path(DATA_PROCESSED),
    splits_csv=splits_csv,
    split="test",
    crop_size=256,
    jitter_radius=20,
    pos_fraction=0.7,
    return_meta=False,
)

print(len(train_ds), len(val_ds), len(test_ds))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=8, pin_memory=True)


In [ ]:
imgs, masks = next(iter(train_loader))
print(imgs.shape, imgs.dtype)   # (B, 8, 256, 256) float32
print(masks.shape, masks.dtype) # (B, 256, 256) int64
print(masks.unique())           # tensor([0, 1])

In [ ]:
pos_ratio = (masks.sum(dim=(1,2)) > 0).float().mean().item()
print("Batch positive sample ratio:", pos_ratio)

In [ ]:
pos_pixels = masks.sum().item()
total_pixels = masks.numel()
print("Positive pixel ratio:", pos_pixels / total_pixels)


In [ ]:
import matplotlib.pyplot as plt

i = 15
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(imgs[i, 0], cmap="gray")
plt.title("Image band 0")

plt.subplot(1,2,2)
plt.imshow(masks[i], cmap="gray")
plt.title("Mask")
plt.show()


##### Removing Flagged scenes

In [ ]:
from pathlib import Path
from config import DATA_PROCESSED
import pandas as pd

macro_csv = Path(DATA_PROCESSED) / "cropped_images/macro_catalog.csv"
flagged_csv = Path(DATA_PROCESSED) / "cropped_images/flagged_macro.csv"
macro_filtered_csv = Path(DATA_PROCESSED) / "cropped_images/macro_catalog_filtered.csv"

df_filtered = filter_macro_catalog(macro_csv, flagged_csv, out_csv=macro_filtered_csv)


print (flagged_csv)
print("Before:", len(pd.read_csv(macro_csv)))
print("After :", len(df_filtered))


##### Spliting data after removing flagged scenes

In [ ]:
from pathlib import Path
from config import DATA_PROCESSED
import pandas as pd

macro_csv = Path(DATA_PROCESSED) / "cropped_images/macro_catalog_filtered.csv"
out_csv = Path(DATA_PROCESSED) / "site_splits.csv"

splits_df = make_site_splits(
    macro_catalog_csv=macro_csv,
    out_splits_csv=out_csv,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42
)

print(splits_df["split"].value_counts())

# Verify how many rows per split (months)
df = pd.read_csv(macro_csv)
df2 = df.merge(splits_df, on="order", how="left")
print(df2["split"].value_counts())

## Crop images Stats

In [ ]:
from patches import compute_train_percentiles, save_percentiles_json
macro_csv  = Path(DATA_PROCESSED) / "cropped_images/macro_catalog_filtered.csv"
splits_csv = Path(DATA_PROCESSED) / "site_splits.csv"
bands = list(range(8))  # or your bands_to_use
stats = compute_train_percentiles(
    macro_catalog_csv=macro_csv,
    site_splits_csv=splits_csv,
    bands_to_use=bands,
    max_images=None,  # optional speedup; set None to use all train images
)
save_percentiles_json(stats, Path(DATA_PROCESSED) / "norm" / "p2_p98_train.json")


In [ ]:
Path(DATA_PROCESSED) / "norm" / "p2_p98_train.json"